# F1 Technical Innovation Classifier - 2025 Alignment Pipeline

**Task 1:** Build the chronological performance tracker `df_performance` using 2025 Pace Gaps & Points.
**Task 2:** Build the alignment engine that applies a 3-race rolling window to map unstructured text articles to a binary success label.

### Cell 1: Setup & API Configurations

In [ ]:
import pandas as pd
import requests
import fastf1
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# FastF1 Configurations
fastf1.Cache.enable_cache('./f1_cache')
import logging
fastf1.logger.set_log_level(logging.ERROR)

# Emergency Rate-Limit Bypass (FastF1 internal limit is 500 API calls/h)
import fastf1.req
if hasattr(fastf1.req, '_CallsPerIntervalLimitRaise'):
    fastf1.req._CallsPerIntervalLimitRaise.limit = lambda self: None

### Cell 2: Task 1 - Generate df_performance (2025 Only)
We pull both Pace Gap (FastF1) and Constructor Points (Jolpica) strictly for the 2025 season to create our performance ledger.

In [ ]:
def fetch_2025_performance():
    season = 2025
    pace_records = []
    
    # --- 1. FASTF1 PACE GAPS ---
    try:
        schedule = fastf1.get_event_schedule(season)
        events = schedule[schedule['EventFormat'] != 'testing']
        
        for _, event in events.iterrows():
            race_date = pd.to_datetime(event['EventDate']).normalize()
            try:
                session = fastf1.get_session(season, event['EventName'], 'Q')
                session.load(telemetry=False, weather=False, messages=False)
                
                pole_time = session.laps['LapTime'].min()
                if pd.isnull(pole_time): continue
                
                teams = session.laps['Team'].dropna().unique()
                for team in teams:
                    team_laps = session.laps[session.laps['Team'] == team]
                    team_fastest = team_laps['LapTime'].min()
                    
                    gap_sec = (team_fastest - pole_time).total_seconds()
                    pace_records.append({'race_date': race_date, 'team': team, 'pace_gap': gap_sec})
            except Exception:
                pass # Catch future races/unavailable sessions
    except Exception as e:
        print(f"FastF1 Error: {e}")
        
    df_pace = pd.DataFrame(pace_records, columns=['race_date', 'team', 'pace_gap'])
    
    # --- 2. JOLPICA CONSTRUCTOR POINTS (WITH PAGINATION) ---
    points_records = []
    offset = 0
    while True:
        url = f"http://api.jolpi.ca/ergast/f1/{season}/results.json?limit=100&offset={offset}"
        try:
            res = requests.get(url)
            if res.status_code == 200:
                data = res.json()['MRData']
                races = data['RaceTable']['Races']
                if not races:
                    break
                
                for race in races:
                    r_date = pd.to_datetime(race['date']).normalize()
                    for result in race['Results']:
                        points_records.append({
                            'race_date': r_date,
                            'team': result['Constructor']['name'],
                            'points': float(result['points'])
                        })
                        
                offset += 100
                if offset >= int(data['total']):
                    break
            else:
                break
        except Exception as e:
            print(f"API Error: {e}")
            break
            
    df_raw_points = pd.DataFrame(points_records, columns=['race_date', 'team', 'points'])
    if not df_raw_points.empty:
        df_points = df_raw_points.groupby(['race_date', 'team'])['points'].sum().reset_index()
        df_points.rename(columns={'points': 'constructor_points'}, inplace=True)
    else:
        df_points = pd.DataFrame(columns=['race_date', 'team', 'constructor_points'])

    # --- 3. MERGE PERFORMANCE LEDGER ---
    team_mapping = {
        'Alpine': 'Alpine F1 Team',
        'Racing Bulls': 'RB F1 Team',
        'Red Bull Racing': 'Red Bull',
        'Kick Sauber': 'Sauber'
    }
    df_pace['team'] = df_pace['team'].replace(team_mapping)
    
    df_perf = pd.merge(df_pace, df_points, on=['race_date', 'team'], how='outer')
    df_perf.dropna(subset=['constructor_points'], inplace=True)
    df_perf.sort_values(by='race_date', inplace=True)
    
    return df_perf

df_performance = fetch_2025_performance()
print(f"df_performance shape: {df_performance.shape}")
display(df_performance.head())

### Cell 3: Task 2 - Alignment Engine (Rolling Window)
We process the unstructured text table (`df_articles`) and compute the binary `is_successful` target column based on a 3-race before/after trend analysis.

In [ ]:
# 1. Setup Dummy Unstructured Text Table (df_articles)
dummy_articles = {
    'date': ['2025-03-23', '2025-04-13'], # Sample 2025 dates
    'team': ['Mercedes', 'Ferrari'],
    'clean_text': [
        "mercedes brought a major front wing upgrade to improve high-speed downforce", 
        "ferrari floor tweaks aimed at reducing porpoising issues"
    ]
}
df_articles = pd.DataFrame(dummy_articles)
df_articles['date'] = pd.to_datetime(df_articles['date'])

# 2. Rolling Window Tracking Function
def map_upgrade_success(article_date, target_team, df_perf, window=3):
    """
    Looks 3 races back and 3 races forward from the article date.
    Returns 1 if Pace Gap shrinks (improves) or points go up. Returns 0 otherwise.
    """
    team_perf = df_perf[df_perf['team'] == target_team]
    
    # Filter Before and After windows
    before = team_perf[team_perf['race_date'] <= article_date].tail(window)
    after = team_perf[team_perf['race_date'] > article_date].head(window)
    
    # If there are no historical or future races to compare, we return NaN (to be dropped in inner join)
    if before.empty or after.empty:
        return np.nan 
        
    # Calculate averages
    avg_gap_before = before['pace_gap'].mean()
    avg_gap_after = after['pace_gap'].mean()
    
    # Primary Metric check: Is the gap to pole smaller?
    if not pd.isna(avg_gap_before) and not pd.isna(avg_gap_after):
        return 1 if avg_gap_after < avg_gap_before else 0
        
    # Fallback Metric check: Did the average points increase?
    avg_pts_before = before['constructor_points'].mean()
    avg_pts_after = after['constructor_points'].mean()
    return 1 if avg_pts_after > avg_pts_before else 0

# 3. Execute Alignment
df_articles['is_successful'] = df_articles.apply(
    lambda row: map_upgrade_success(row['date'], row['team'], df_performance), axis=1
)

# 4. Inner Merge Filter (Drop unverified texts)
df_features_labels = df_articles.dropna(subset=['is_successful']).copy()
df_features_labels['is_successful'] = df_features_labels['is_successful'].astype(int)

# 5. Export The Machine Learning Target Matrix
df_features_labels.to_csv('master_dataset.csv', index=False)
print(f"df_features_labels shape: {df_features_labels.shape}")
display(df_features_labels)